# 📧 Email Content Quality Scoring — Model Selection (Google Colab)

Predict whether an email gets a **reply** from its content, and pick the best model.

**Dataset:** `email_replied_dataset.csv` — 1106 balanced rows (`replied` = 0/1).

Features = raw text (`subject`, `body` → **TF-IDF**) + engineered signals (`word_count`, `has_question`, `has_cta`, `has_link`, `personalization_level`, `num_exclamations`) + categoricals (`tone`, `length_bucket`).

We benchmark **9 models** with 5-fold cross-validation, report full metrics, plot **ROC** + **Precision-Recall** curves and feature importances, then save the winner.

**How to run:** *Runtime ▸ Run all* — the second cell opens a file picker; upload the CSV.

## 0 · Setup

In [ ]:
# Colab usually ships these, but pin them just in case:
!pip -q install lightgbm xgboost scikit-learn pandas matplotlib

In [ ]:
import warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, confusion_matrix, classification_report)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
plt.rcParams["figure.dpi"] = 110

## 1 · Load (upload CSV)

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────
# In Colab this opens a file picker — upload email_replied_dataset.csv.
try:
    from google.colab import files
    uploaded = files.upload()
    DATA = list(uploaded.keys())[0]
except Exception:
    DATA = "email_replied_dataset.csv"   # local fallback

df = pd.read_csv(DATA)
print("Rows:", len(df))
print("Class balance:\n", df["replied"].value_counts())
df.head(3)

## 2 · Features

In [ ]:
# ── Feature groups ────────────────────────────────────────────────────
# text -> TF-IDF   |   numeric/binary -> scaled   |   tone/length -> one-hot
TEXT_COL = "full_text"
NUM_COLS = ["word_count", "has_question", "has_cta", "has_link",
            "personalization_level", "num_exclamations"]
CAT_COLS = ["tone", "length_bucket"]

df = df.dropna(subset=["replied"]).reset_index(drop=True)
df["subject"] = df["subject"].fillna("")
df["body"]    = df["body"].fillna("")
df[TEXT_COL]  = df["subject"].astype(str) + " . " + df["body"].astype(str)
for col in NUM_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

X = df[[TEXT_COL] + NUM_COLS + CAT_COLS]
y = df["replied"].astype(int).values
print("Features:", NUM_COLS, "+ TF-IDF +", CAT_COLS)
X.head(3)

In [ ]:
# ── Split + shared preprocessing ──────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

pre = ColumnTransformer(
    transformers=[
        ("txt", TfidfVectorizer(max_features=1500, ngram_range=(1, 2),
                                stop_words="english", min_df=2), TEXT_COL),
        ("num", MinMaxScaler(), NUM_COLS),                       # >= 0 for NB
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ],
    sparse_threshold=0,   # dense -> every model below is compatible
)
print("Train:", X_train.shape[0], " Test:", X_test.shape[0])

## 3 · Candidate models

In [ ]:
# ── Candidate models ──────────────────────────────────────────────────
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, C=1.0),
    "ComplementNB":       ComplementNB(),
    "LinearSVC (calib.)": CalibratedClassifierCV(LinearSVC(max_iter=5000)),
    "KNN":                KNeighborsClassifier(n_neighbors=7),
    "RandomForest":       RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE),
    "ExtraTrees":         ExtraTreesClassifier(n_estimators=400, random_state=RANDOM_STATE),
    "GradientBoosting":   GradientBoostingClassifier(random_state=RANDOM_STATE),
    "LightGBM":           LGBMClassifier(random_state=RANDOM_STATE, verbose=-1),
    "XGBoost":            XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss"),
}

## 4 · Train & evaluate

In [ ]:
# ── Train, cross-validate and evaluate every model ────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results, curves = [], {}

for name, clf in models.items():
    pipe = Pipeline([("pre", pre), ("clf", clf)])
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc").mean()
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    results.append({
        "Model":      name,
        "CV ROC-AUC": round(cv_auc, 3),
        "ROC-AUC":    round(roc_auc_score(y_test, proba), 3),
        "Accuracy":   round(accuracy_score(y_test, pred), 3),
        "Precision":  round(precision_score(y_test, pred, zero_division=0), 3),
        "Recall":     round(recall_score(y_test, pred, zero_division=0), 3),
        "F1":         round(f1_score(y_test, pred, zero_division=0), 3),
        "AvgPrec":    round(average_precision_score(y_test, proba), 3),
    })
    curves[name] = proba

res = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
res

## 5 · Curves & comparison

In [ ]:
# ── Metric comparison (bar chart) ─────────────────────────────────────
metrics = ["ROC-AUC", "Accuracy", "F1", "Precision", "Recall"]
order = res["Model"].tolist()
xpos = np.arange(len(order)); w = 0.15
fig, ax = plt.subplots(figsize=(12, 5))
for i, m in enumerate(metrics):
    ax.bar(xpos + i*w, res.set_index("Model").loc[order, m], w, label=m)
ax.set_xticks(xpos + w*2); ax.set_xticklabels(order, rotation=25, ha="right")
ax.set_ylim(0, 1); ax.set_ylabel("Score"); ax.set_title("Model comparison")
ax.legend(ncol=5, loc="lower center", bbox_to_anchor=(0.5, -0.32)); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── ROC curves (all models) ───────────────────────────────────────────
plt.figure(figsize=(7, 6))
for name in order:
    fpr, tpr, _ = roc_curve(y_test, curves[name])
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, curves[name]):.2f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC curves"); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Precision-Recall curves (all models) ──────────────────────────────
plt.figure(figsize=(7, 6))
for name in order:
    prec, rec, _ = precision_recall_curve(y_test, curves[name])
    plt.plot(rec, prec, label=f"{name} (AP={average_precision_score(y_test, curves[name]):.2f})")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("Precision-Recall curves"); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6 · Best model

In [ ]:
# ── Best model: confusion matrix + report ─────────────────────────────
best_name = res.iloc[0]["Model"]
best_pipe = Pipeline([("pre", pre), ("clf", models[best_name])]).fit(X_train, y_train)
best_pred = best_pipe.predict(X_test)
print("Best model:", best_name)
print(classification_report(y_test, best_pred, target_names=["no reply", "replied"]))

cm = confusion_matrix(y_test, best_pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=14)
ax.set_xticks([0, 1]); ax.set_xticklabels(["no reply", "replied"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["no reply", "replied"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"Confusion matrix — {best_name}")
plt.tight_layout(); plt.show()

In [ ]:
# ── Top feature importances of the best model (if available) ──────────
try:
    names = best_pipe.named_steps["pre"].get_feature_names_out()
    clf = best_pipe.named_steps["clf"]
    imp = getattr(clf, "feature_importances_", None)
    if imp is None and hasattr(clf, "coef_"):
        imp = np.abs(clf.coef_).ravel()
    if imp is not None:
        top = np.argsort(imp)[-20:]
        plt.figure(figsize=(8, 6))
        plt.barh(range(len(top)), imp[top])
        plt.yticks(range(len(top)), [names[i] for i in top], fontsize=8)
        plt.title(f"Top 20 features — {best_name}")
        plt.tight_layout(); plt.show()
    else:
        print("This model exposes no feature importances.")
except Exception as e:
    print("Importance plot skipped:", e)

## 7 · Save & demo

In [ ]:
# ── Retrain best on ALL data and save ─────────────────────────────────
final = Pipeline([("pre", pre), ("clf", models[best_name])]).fit(X, y)
joblib.dump({"pipeline": final, "num_cols": NUM_COLS, "cat_cols": CAT_COLS, "text_col": TEXT_COL},
            "email_quality_model.joblib")
print("Saved -> email_quality_model.joblib  (best =", best_name, ")")

# In Colab, download it to your machine:
try:
    from google.colab import files
    files.download("email_quality_model.joblib")
except Exception:
    pass

In [ ]:
# ── Demo: score a new email draft ─────────────────────────────────────
def quality_score(subject, body, word_count, has_question, has_cta, has_link,
                  personalization_level, num_exclamations, tone, length_bucket):
    row = pd.DataFrame([{
        "full_text": f"{subject} . {body}",
        "word_count": word_count, "has_question": has_question, "has_cta": has_cta,
        "has_link": has_link, "personalization_level": personalization_level,
        "num_exclamations": num_exclamations, "tone": tone, "length_bucket": length_bucket,
    }])
    return round(float(final.predict_proba(row)[:, 1][0]) * 100, 1)

print(quality_score("Idea for your outbound",
                    "Hi Marco, noticed your team focuses on churn. We help with exactly that. Book a slot: https://cal.com/x",
                    52, 0, 1, 1, 2, 0, "formal", "short"), "%  (personalized + CTA + link)")
print(quality_score("AMAZING OFFER!!! ACT NOW",
                    "Dear customer, our unbeatable solution guarantees results. Click now to win!!!",
                    40, 0, 1, 0, 0, 5, "spammy", "medium"), "%  (spammy)")